# Drug Shortages Canada: Data

## Package Imports

General package imports, plus setup for local package import.

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
from time import time
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

Functions for accessing the API imported from local package.

In [ ]:
from Drug_Shortage_CA_Helpers.helpers import (get_drug_data, 
                                              get_company_data, 
                                              get_drug_ingredients, 
                                              get_drug_status, 
                                              get_shortage_data)

Data on the drug products involved.

In [ ]:
drug_data = pd.DataFrame([{"Drug Code": d[0], 
                           "Drug Brand Name": d[1], 
                           "Company Name": d[2], 
                           "DIN": d[3], 
                           "Drug Class": d[4]} for d in get_drug_data()])
drug_data.head(1)

Data on the product owning entities.

In [ ]:
company_data = pd.DataFrame([{"Company Code": d[0], 
                              "Company Name": d[1], 
                              "Company Country Name": d[2]} for d in get_company_data()])
company_data.head(1)

Data on the drug ingredients.

In [ ]:
ingredient_data = pd.DataFrame([{"Drug Code": d[1], 
                                 "Ingredient Name": d[0]} for d in get_drug_ingredients()])
ingredient_data.head(1)

Drug status data.

In [ ]:
drug_statuses = pd.DataFrame([{"Drug Code": d[1], 
                               "Drug Status": d[0]} for d in get_drug_status()])
drug_statuses.head(1)

Drug shortage data - this could take up to 10 mins to be retrieved.

In [7]:
drug_shortages = pd.DataFrame([{"Drug Code": d[0], 
                                "Company Code": d[1], 
                                "Shortage Reason": d[2], 
                                "Shortage Started Date": d[3], 
                                "Report ID": d[4], 
                                "DIN": d[5], 
                                "Shortage Status": d[6]} for d in get_shortage_data(shortage_status=None)])
drug_shortages.head(1)

The dataframes are joined into a single dataframe on the appropriate columns. Note that the resulting dataframe will be one row per ingredient, not one row per drug.

In [ ]:
df = pd.merge(drug_shortages, drug_data, on=["Drug Code", "DIN"], how="left")
df = pd.merge(df, drug_statuses, on="Drug Code", how="left")
df = pd.merge(df, ingredient_data, on="Drug Code", how="inner")
df = pd.merge(df, company_data, on=["Company Code", "Company Name"], how="left")
df.head(1)

The dataframe is saved as a CSV to `notebooks/Outputs`.

In [ ]:
if not os.path.exists("../Outputs"):
    Path("../Outputs").mkdir(parents=True, exist_ok=True)

df.to_csv(f"Outputs/Drug Shortage Data - {str(int(time() * 1000))}.csv", encoding="utf-8-sig", index=False)